# 0. Introduction

> <center> Questo report presenta i risultati di un backtesting completo volto a verificare se il sentiment giornaliero dei commenti su Reddit relativi all'ETF GLD (oro) abbia una correlazione statisticamente significativa con i rendimenti effettivi dell'indice.

---

# 1. Reddit request API: GLD

In [28]:
import praw
import pandas as pd
from datetime import datetime, timedelta

reddit = praw.Reddit(
    client_id="-",
    client_secret="-",
    user_agent="market-sentiment-research by amicopaolo",
    check_for_async=False
)

subreddits = ["Gold", "GoldInvestor", "investing"]
query = '"gold" OR "gold price" OR "gold ETF" OR "XAU" OR "GLD"'
oggi = datetime.utcnow()
dieci_anni_fa = oggi - timedelta(days=365*10)
start_ts = dieci_anni_fa.timestamp()

rows = []

for name in subreddits:
    sub = reddit.subreddit(name)
    for submission in sub.search(query, sort="new", limit=None):
        if submission.created_utc >= start_ts:
            rows.append({
                "subreddit": name,
                "id": submission.id,
                "created_utc": submission.created_utc,
                "title": submission.title,
                "selftext": submission.selftext,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "url": submission.url,
            })
        else:
            break

df = pd.DataFrame(rows)
df.to_csv("reddit_gold_posts_10y.csv", index=False, encoding="utf-8")

> <center> Lo script utilizza le API ufficiali di Reddit tramite la libreria PRAW per estrarre i post relativi all'oro dai subreddit Gold, GoldInvestor e investing. La query di ricerca filtra i post contenenti i termini "gold", "gold price", "gold ETF", "XAU" o "GLD" negli ultimi 10 anni. Per ogni post vengono raccolti: subreddit di provenienza, ID univoco, timestamp di creazione, titolo, corpo del testo, punteggio, numero di commenti e URL. I risultati vengono salvati nel file reddit_gold_posts_10y.csv.

---

# 2. Dataset cleaning

In [ ]:
import pandas as pd
from datetime import datetime, timezone

df = pd.read_csv("reddit_gold_posts_10y.csv")

df["datetime_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)

df["text"] = df["title"].fillna("") + " " + df["selftext"].fillna("")

df_min = df[["subreddit", "datetime_utc", "text", "score", "num_comments"]]

df["len_text"] = df["text"].str.len()
df = df[df["len_text"] >= 50]

df = df[df["score"] >= 1]
df = df[df["num_comments"] >= 1]

spam_patterns = ["whatsapp", "telegram", "signals", "forex broker", "join my group"]
mask_spam = df["text"].str.lower().str.contains("|".join(spam_patterns))
df = df[~mask_spam]

df = df[df["text"].str.strip() != ""]

df_min.to_csv("reddit_gold_posts_10y_clean.csv", index=False, encoding="utf-8")

> <center> Il dataset grezzo viene sottoposto a una serie di filtri di qualità: si concatenano titolo e corpo del testo in un unico campo, si eliminano i post con testo inferiore a 50 caratteri, con punteggio inferiore a 1 o senza commenti. Vengono inoltre rimossi i post contenenti pattern tipici di spam (whatsapp, telegram, signals, forex broker, join my group) e quelli con testo vuoto. Il dataset pulito viene esportato in reddit_gold_posts_10y_clean.csv e contiene le colonne: subreddit, datetime_utc, text, score, num_comments.

---

# 3. Daily sentiment

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import matplotlib.pyplot as plt

df = pd.read_csv("reddit_gold_posts_10y_clean.csv")

df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
df["date"] = df["datetime_utc"].dt.date

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    truncation=True,
    max_length=512,
)

def score_to_polarity(scores):
    if isinstance(scores, dict):
        scores = [scores]

    if isinstance(scores, list) and len(scores) > 0 and isinstance(scores[0], list):
        scores = scores[0]

    if not (isinstance(scores, list) and len(scores) > 0 and isinstance(scores[0], dict)):
        return 0.0

    d = {s["label"].lower(): s["score"] for s in scores}
    pos = d.get("positive", 0.0)
    neg = d.get("negative", 0.0)
    return pos - neg

df_sample = df.copy()

sentiments = []
for text in df_sample["text"]:
    if not isinstance(text, str) or text.strip() == "":
        sentiments.append(0.0)
        continue
    out = finbert(text)
    sentiments.append(score_to_polarity(out))

df_sample["sentiment"] = sentiments

daily = (
    df_sample.groupby("date")["sentiment"]
    .mean()
    .reset_index()
    .sort_values("date")
)

daily.to_csv("daily_gold_sentiment.csv", index=False, encoding="utf-8")

print("\nNumero di post analizzati:", len(df_sample))
print("Numero di giorni nella serie:", len(daily))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Numero di post analizzati: 669
Numero di giorni nella serie: 248


> <center> Il modello FinBERT (ProsusAI/finbert) è stato applicato ai 669 post puliti. Per ogni testo il modello produce tre probabilità (positive, negative, neutral) e il punteggio di polarità è calcolato come differenza tra la probabilità positiva e quella negativa (range −1 / +1). Il sentiment giornaliero è la media dei punteggi dei post pubblicati nello stesso giorno. La serie risultante copre 248 giorni, di cui 82 con sentiment ≠ 0 (almeno un post rilevante). Il file daily_gold_sentiment.csv contiene le colonne date e sentiment.

---

# 4. Finance datas

In [31]:
import datetime as dt
import yfinance as yf

ticker = "GLD"

oggi = dt.date.today()
dieci_anni_fa = oggi - dt.timedelta(days=365 * 10)

start_date = dieci_anni_fa.strftime("%Y-%m-%d")
end_date = oggi.strftime("%Y-%m-%d")

data = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    interval="1d",     
    auto_adjust=False,
    progress=True
)

data.to_csv("yfinance_gold_10y.csv", index=True)

[*********************100%***********************]  1 of 1 completed


> <center> La serie storica di GLD è stata scaricata tramite yfinance con intervallo giornaliero, coprendo gli ultimi 10 anni fino alla data di esecuzione. Il file yfinance_gold_10y.csv contiene le colonne Date, AdjClose, Close, High, Low, Open e Volume. Viene effettuato il merge per data esatta con il file di sentiment e il rendimento forward a ogni orizzonte (1, 3, 5, 10, 15, 20, 30, 40, 60 giorni lavorativi) è calcolato come variazione percentuale del Close, shiftata in avanti per evitare look-ahead bias: il sentiment al giorno t è sempre accoppiato con il rendimento da t+1 in poi.

---

# 5. Analysis

In [32]:
import pandas as pd
import numpy as np
from scipy import stats

sent = pd.read_csv("daily_gold_sentiment.csv", parse_dates=["date"])
sent.columns = ["date", "sentiment"]

prices_raw = pd.read_csv("yfinance_gold_10y.csv")
prices_raw.columns = ["Date", "AdjClose", "Close", "High", "Low", "Open", "Volume"]
prices = prices_raw.iloc[2:].copy()
prices["Date"] = pd.to_datetime(prices["Date"])
for c in ["AdjClose", "Close", "High", "Low", "Open", "Volume"]:
    prices[c] = pd.to_numeric(prices[c], errors="coerce")
prices = prices.sort_values("Date").reset_index(drop=True)

horizons = [1, 3, 5, 10, 15, 20, 30, 40, 60]
for k in horizons:
    prices[f"ret_{k}d"] = prices["Close"].pct_change(k).shift(-k)

df = pd.merge(sent, prices, left_on="date", right_on="Date", how="inner")
df = df.sort_values("date").reset_index(drop=True)
df_nz = df[df["sentiment"] != 0].copy()


> <center> Vengono utilizzati i seguenti file risultanti: <b>daily_gold_sentiment.csv</b> e <b>yfinance_gold_10y.csv</b>. <br>
> <center> Viene effettuato il merge per data e il sentiment al giorno t è sempre accoppiato con il rendimento da t+1 in poi, per evitare qualsiasi look-ahead bias. 

## 5.1. Correlazioni Person e Spearman

In [ ]:
results_corr = []
for k in horizons:
    col = f"ret_{k}d"
    for name, subset_base in [("Tutti", df), ("Sent!=0", df_nz)]:
        subset = subset_base[["sentiment", col]].dropna()
        if len(subset) < 5:
            continue
        pr, pp = stats.pearsonr(subset["sentiment"], subset[col])
        sr, sp = stats.spearmanr(subset["sentiment"], subset[col])
        results_corr.append({
            "Orizzonte": f"{k}d",
            "Campione": name,
            "N": len(subset),
            "Pearson_r": round(pr, 4),
            "Pearson_p": round(pp, 4),
            "Spearman_r": round(sr, 4),
            "Spearman_p": round(sp, 4),
        })

df_corr = pd.DataFrame(results_corr)
df_corr

,Orizzonte,Campione,N,Pearson_r,Pearson_p,Spearman_r,Spearman_p
0,1d,Tutti,197,-0.0554,0.4393,-0.0922,0.1973
1,1d,Sent!=0,82,0.0105,0.9255,-0.0897,0.4228
2,3d,Tutti,195,-0.0101,0.8884,-0.0303,0.6744
3,3d,Sent!=0,80,0.0858,0.4491,0.0330,0.7716
4,5d,Tutti,193,-0.0444,0.5395,-0.0169,0.8153
5,5d,Sent!=0,79,0.0701,0.5392,0.0572,0.6167
6,10d,Tutti,189,-0.0723,0.3231,-0.0206,0.7780
7,10d,Sent!=0,77,-0.0145,0.9002,0.0392,0.7348
8,15d,Tutti,184,-0.1037,0.1611,-0.0566,0.4450
9,15d,Sent!=0,76,-0.0278,0.8115,0.0376,0.7473


> <center> Nel breve termine (1–20 giorni) nessuna correlazione raggiunge la significatività: tutti i p-value superano 0.12, e i coefficienti oscillano attorno allo zero senza direzione coerente (il più alto in valore assoluto è Pearson r = −0.116 a 20 giorni con p = 0.120). Nel lungo termine (30–60 giorni) la correlazione resta negativa sul campione completo — a 30 giorni r = −0.148 (p = 0.053), a 40 giorni r = −0.154 (p = 0.051) — ma non raggiunge mai la soglia convenzionale del 5%. Sul sottocampione Sent≠0, i valori sono ancora più deboli: a 60 giorni r = −0.092 (p = 0.471). A differenza della precedente esecuzione, nessun orizzonte risulta statisticamente significativo: il trend negativo è presente ma troppo debole per essere distinto dal rumore, con il campione attuale di 248 giorni.

## 5.2. Regressione OLS

In [ ]:
results_reg = []
for k in horizons:
    col = f"ret_{k}d"
    for name, subset_base in [("Tutti", df), ("Sent!=0", df_nz)]:
        subset = subset_base[["sentiment", col]].dropna()
        if len(subset) < 5:
            continue
        sl, it, rv, pv, se = stats.linregress(subset["sentiment"], subset[col])
        results_reg.append({
            "Orizzonte": f"{k}d",
            "Campione": name,
            "N": len(subset),
            "Beta": round(sl, 6),
            "p_beta": round(pv, 4),
            "R2": round(rv**2, 4),
        })

df_reg = pd.DataFrame(results_reg)
df_reg

,Orizzonte,Campione,N,Beta,p_beta,R2
0,1d,Tutti,197,-0.002233,0.4393,0.0031
1,1d,Sent!=0,82,0.000338,0.9255,0.0001
2,3d,Tutti,195,-0.000675,0.8884,0.0001
3,3d,Sent!=0,80,0.004834,0.4491,0.0074
4,5d,Tutti,193,-0.003605,0.5395,0.0020
5,5d,Sent!=0,79,0.004019,0.5392,0.0049
6,10d,Tutti,189,-0.007058,0.3231,0.0052
7,10d,Sent!=0,77,-0.001090,0.9002,0.0002
8,15d,Tutti,184,-0.013277,0.1611,0.0108
9,15d,Sent!=0,76,-0.001905,0.8115,0.0008


> <center> Nel breve termine il coefficiente β non è mai significativo e l'R² resta sotto l'1.4%: il sentiment Reddit non spiega praticamente nulla della varianza dei rendimenti futuri di GLD da 1 a 20 giorni. Nel lungo termine β è sistematicamente negativo sul campione completo (β = −0.027 a 30d, β = −0.030 a 40d, β = −0.027 a 60d) ma nessun p-value scende sotto 0.05: il più basso è p = 0.051 a 40 giorni con R² = 2.4%. Sul sottocampione Sent≠0 l'effetto è ancora più debole, con R² massimo dello 0.85% a 60 giorni (p = 0.471). In questa esecuzione aggiornata, la regressione conferma un'indicazione direzionale negativa — più ottimismo su Reddit tende ad associarsi a rendimenti leggermente inferiori — ma la relazione non è statisticamente robusta.

## 5.3. Analisi Lead-Lag

In [ ]:
results_lag = []
for lag in range(-15, 16):
    temp = df[["sentiment", "ret_1d"]].copy()
    temp["sent_lag"] = temp["sentiment"].shift(lag)
    temp = temp[["sent_lag", "ret_1d"]].dropna()
    temp_nz = temp[temp["sent_lag"] != 0]
    if len(temp_nz) >= 5:
        r, p = stats.pearsonr(temp_nz["sent_lag"], temp_nz["ret_1d"])
        results_lag.append({
            "Lag": lag,
            "Pearson_r": round(r, 4),
            "p_value": round(p, 4),
            "N": len(temp_nz),
        })

df_lag = pd.DataFrame(results_lag)
df_lag

,Lag,Pearson_r,p_value,N
0,-15,0.0093,0.9380,73
1,-14,-0.0533,0.6518,74
2,-13,0.0698,0.5543,74
3,-12,0.0021,0.9858,74
4,-11,0.0041,0.9720,75
5,-10,-0.1227,0.2912,76
6,-9,0.0278,0.8104,77
7,-8,-0.0056,0.9618,77
8,-7,-0.1423,0.2139,78
9,-6,-0.1746,0.1239,79


> <center> L'unico segnale significativo dell'intera analisi si trova a lag = −3 (r = −0.314, p = 0.004), più forte rispetto alla precedente esecuzione (era r = −0.251, p = 0.021). Questo indica una relazione inversa: il rendimento di 3 giorni prima è correlato negativamente con il sentiment di oggi. In altre parole, il mercato guida il sentiment di Reddit — dopo un calo di GLD, Reddit diventa più negativo 3 giorni dopo — e non il contrario. I lag positivi (sentiment che anticipa i rendimenti) non mostrano alcuna significatività: tutti i p-value sono ampiamente sopra 0.10, confermando che il sentiment Reddit non ha potere predittivo giornaliero. Anche estendendo i lag fino a ±15 giorni non emergono nuovi segnali significativi.

## 5.4. Analisi Event-Based

In [ ]:
q30 = df_nz["sentiment"].quantile(0.30)
q70 = df_nz["sentiment"].quantile(0.70)

df_nz["regime"] = "Neutro"
df_nz.loc[df_nz["sentiment"] >= q70, "regime"] = "HighBull"
df_nz.loc[df_nz["sentiment"] <= q30, "regime"] = "HighBear"

event_results = []
for regime in ["HighBull", "HighBear", "Neutro"]:
    sub = df_nz[df_nz["regime"] == regime]
    for k in horizons:
        col = f"ret_{k}d"
        vals = sub[col].dropna()
        if len(vals) < 3:
            continue
        t_stat, t_p = stats.ttest_1samp(vals, 0)
        event_results.append({
            "Regime": regime,
            "Orizzonte": f"{k}d",
            "N": len(vals),
            "Ret_medio_%": round(vals.mean() * 100, 3),
            "t_stat": round(t_stat, 3),
            "p_value": round(t_p, 4),
        })

df_events = pd.DataFrame(event_results)
df_events

,Regime,Orizzonte,N,Ret_medio_%,t_stat,p_value
0,HighBull,1d,25,-0.296,-1.278,0.2134
1,HighBull,3d,25,-0.082,-0.229,0.8206
2,HighBull,5d,25,-0.230,-0.462,0.6481
3,HighBull,10d,25,0.125,0.226,0.8230
4,HighBull,15d,25,0.106,0.171,0.8657
5,HighBull,20d,25,0.195,0.328,0.7455
6,HighBull,30d,25,0.130,0.149,0.8831
7,HighBull,40d,25,0.033,0.032,0.9751
8,HighBull,60d,25,1.357,1.091,0.2861
9,HighBear,1d,25,-0.407,-0.742,0.4650


> <center> Nel breve termine (1–10 giorni) nessun regime produce rendimenti medi statisticamente diversi da zero: sia i giorni molto bullish sia quelli molto bearish generano risultati negativi (HighBull: −0.30% a 1d, −0.23% a 5d; HighBear: −0.41% a 1d, −1.06% a 5d), e nessun t-test raggiunge la significatività (tutti i p > 0.21). Nel lungo termine (30–60 giorni) il regime HighBear mostra rendimenti positivi (+0.40% a 30d, +1.67% a 40d, +2.25% a 60d) ma con p-value elevati (p = 0.788 a 30d, p = 0.384 a 40d, p = 0.286 a 60d): nessuno è statisticamente significativo. Il regime HighBull mostra rendimenti positivi solo a 60 giorni (+1.36%, p = 0.286). L'effetto contrarian che nella precedente esecuzione era visibile (HighBear a +5.68% a 60d) risulta qui notevolmente ridimensionato: il Bear continua a sovraperformare il Bull sugli orizzonti lunghi, ma la differenza è molto più contenuta e priva di significatività statistica.

## 5.5. T-Test HighBull vs HighBear

In [ ]:
ttest_results = []
for k in horizons:
    col = f"ret_{k}d"
    bv = df_nz[df_nz["regime"] == "HighBull"][col].dropna()
    brv = df_nz[df_nz["regime"] == "HighBear"][col].dropna()
    if len(bv) >= 3 and len(brv) >= 3:
        t, p = stats.ttest_ind(bv, brv, equal_var=False)
        ttest_results.append({
            "Orizzonte": f"{k}d",
            "Bull_mean_%": round(bv.mean() * 100, 3),
            "Bear_mean_%": round(brv.mean() * 100, 3),
            "Diff_%": round((bv.mean() - brv.mean()) * 100, 3),
            "t_stat": round(t, 3),
            "p_value": round(p, 4),
        })

df_ttest = pd.DataFrame(ttest_results)
df_ttest

,Orizzonte,Bull_mean_%,Bear_mean_%,Diff_%,t_stat,p_value
0,1d,-0.296,-0.407,0.111,0.187,0.8528
1,3d,-0.082,-0.834,0.752,0.745,0.4620
2,5d,-0.230,-1.056,0.826,0.843,0.4049
3,10d,0.125,-0.679,0.804,0.814,0.4210
4,15d,0.106,-0.030,0.136,0.125,0.9014
5,20d,0.195,0.609,-0.415,-0.295,0.7699
6,30d,0.130,0.395,-0.264,-0.157,0.8767
7,40d,0.033,1.671,-1.638,-0.767,0.4508
8,60d,1.357,2.254,-0.897,-0.378,0.7088


> <center> Nel breve termine (1–10 giorni) la differenza tra Bull e Bear è minima e mai significativa (p > 0.40 per tutti gli orizzonti): il sentiment estremo non discrimina i rendimenti a pochi giorni. Nel lungo termine, il HighBear continua a sovraperformare leggermente il HighBull — a 40 giorni la differenza è di −1.64 punti percentuali, a 60 giorni di −0.90 punti — ma i p-value sono molto alti (p = 0.45 a 40d, p = 0.70 a 60d), ben lontani dalla significatività. Con 25 osservazioni per il gruppo Bull e 13–25 per il gruppo Bear a seconda dell'orizzonte, la potenza statistica è del tutto insufficiente. Rispetto alla precedente esecuzione, dove la differenza a 60 giorni era di −4.16 punti percentuali (p = 0.191), i nuovi dati riducono l'effetto a meno di un quarto e lo rendono ancora meno distinguibile dal rumore casuale.

## 5.6. Backtest Strategie

In [ ]:
q30 = df_nz["sentiment"].quantile(0.30)
q70 = df_nz["sentiment"].quantile(0.70)

df_strat = df.copy().dropna(subset=["ret_1d"])

df_strat["sig1"] = 0
df_strat.loc[(df_strat["sentiment"] >= q70) & (df_strat["sentiment"] != 0), "sig1"] = 1
df_strat["ret_s1"] = df_strat["sig1"] * df_strat["ret_1d"]
df_strat["cum_s1"] = (1 + df_strat["ret_s1"]).cumprod()
df_strat["cum_bh"] = (1 + df_strat["ret_1d"]).cumprod()

df_strat["sig2"] = 0
df_strat.loc[(df_strat["sentiment"] >= q70) & (df_strat["sentiment"] != 0), "sig2"] = 1
df_strat.loc[(df_strat["sentiment"] <= q30) & (df_strat["sentiment"] != 0), "sig2"] = -1
df_strat["ret_s2"] = df_strat["sig2"] * df_strat["ret_1d"]
df_strat["cum_s2"] = (1 + df_strat["ret_s2"]).cumprod()

def calc_metrics(rets, cum):
    total = cum.iloc[-1] - 1
    sharpe = rets.mean() / rets.std() * np.sqrt(252) if rets.std() > 0 else 0
    dd = (cum / cum.cummax() - 1).min()
    return total, sharpe, dd

s1_tot, s1_sh, s1_dd = calc_metrics(df_strat["ret_s1"], df_strat["cum_s1"])
s2_tot, s2_sh, s2_dd = calc_metrics(df_strat["ret_s2"], df_strat["cum_s2"])
bh_tot, bh_sh, bh_dd = calc_metrics(df_strat["ret_1d"], df_strat["cum_bh"])

n_s1 = (df_strat["sig1"] == 1).sum()
n_s2_l = (df_strat["sig2"] == 1).sum()
n_s2_s = (df_strat["sig2"] == -1).sum()

backtest = pd.DataFrame({
    "Strategia": ["Long Sent>Q70", "Long/Short", "Buy & Hold"],
    "Ret_tot_%": [round(s1_tot*100, 2), round(s2_tot*100, 2), round(bh_tot*100, 2)],
    "Sharpe": [round(s1_sh, 3), round(s2_sh, 3), round(bh_sh, 3)],
    "MaxDD_%": [round(s1_dd*100, 2), round(s2_dd*100, 2), round(bh_dd*100, 2)],
    "N_trades": [n_s1, n_s2_l + n_s2_s, len(df_strat)],
})
backtest

,Strategia,Ret_tot_%,Sharpe,MaxDD_%,N_trades
0,Long Sent>Q70,-7.29,-1.430,-7.29,25
1,Long/Short,1.74,0.212,-10.79,50
2,Buy & Hold,19.95,1.080,-13.87,197


> <center> Le strategie basate sul sentiment operano nel breve termine (entrata e uscita giornaliera), dove il segnale predittivo è assente. La strategia "Long solo quando Reddit è bullish" produce un rendimento negativo (−7.29% con Sharpe di −1.430), coerente con l'idea che seguire l'euforia di Reddit è controproducente. La strategia Long/Short genera un modesto +1.74% ma con Sharpe molto basso (0.212 vs 1.080 del Buy & Hold) e senza considerare i costi di transazione. Il Buy & Hold su GLD nello stesso periodo rende +19.95% con Sharpe di 1.080, sovraperformando ampiamente entrambe le strategie basate sul sentiment. Alla luce dell'assenza di significatività statistica anche sugli orizzonti lunghi, l'utilità operativa del sentiment Reddit come segnale di trading per GLD appare molto limitata con i dati attuali.

## 5.7. Rolling Correlation

In [ ]:
df_roll = df[["date", "sentiment", "ret_5d"]].dropna().copy()
df_roll = df_roll.sort_values("date").reset_index(drop=True)

rolling_corrs = []
window = 30
for i in range(window, len(df_roll)):
    chunk = df_roll.iloc[i - window:i]
    chunk_nz = chunk[chunk["sentiment"] != 0]
    if len(chunk_nz) >= 5:
        r, p = stats.pearsonr(chunk_nz["sentiment"], chunk_nz["ret_5d"])
        rolling_corrs.append({
            "date": df_roll.iloc[i]["date"],
            "rolling_r": r,
            "p_value": p,
            "n_nz": len(chunk_nz),
        })

df_rolling = pd.DataFrame(rolling_corrs)
print(f"N finestre: {len(df_rolling)}")
print(f"Rolling r: min={df_rolling['rolling_r'].min():.3f}, max={df_rolling['rolling_r'].max():.3f}")
print(f"Finestre con p<0.05: {(df_rolling['p_value'] < 0.05).sum()}/{len(df_rolling)}")


N finestre: 162
Rolling r: min=-0.657, max=0.916
Finestre con p<0.05: 2/162


> <center> La correlazione rolling (finestra 30 giorni, sentiment vs rendimento 5d) oscilla tra −0.657 e +0.916, con 2 finestre su 162 (1.2%) che raggiungono la significatività (p < 0.05). Nel breve termine la relazione sentiment-rendimento non è stabile: ci sono fasi in cui sembra positiva, fasi in cui si inverte e fasi in cui scompare del tutto. Il range estremamente ampio (da −0.66 a +0.92) conferma che il nesso è dominato dal rumore campionario. Questa instabilità è un forte segnale di cautela contro l'utilizzo operativo del sentiment Reddit per strategie a qualsiasi orizzonte.

---

# 6. Conclusions

> <center> Né nel breve termine (1–20 giorni) né nel lungo termine (30–60 giorni) emerge una relazione statisticamente significativa tra il sentiment di Reddit e i rendimenti futuri di GLD. I p-value più bassi (0.051–0.053 a 30–40 giorni sul campione completo) sfiorano la soglia del 5% ma non la superano, e sul sottocampione Sent≠0 l'effetto si attenua ulteriormente. L'unico risultato significativo è il lead-lag a lag = −3 (r = −0.314, p = 0.004), che dimostra che Reddit reagisce al mercato con circa 3 giorni di ritardo.

> <center> Il segnale contrarian è reale? Con i dati aggiornati (248 giorni di sentiment, 669 post analizzati), il segnale contrarian che emergeva nella precedente esecuzione (r = −0.275, p = 0.026 a 60d) non è confermato. La direzione negativa della correlazione è coerente, ma la significatività è scomparsa. Due interpretazioni restano valide: (a) il segnale era un falso positivo amplificato da un campione leggermente diverso, oppure (b) l'effetto esiste ma è troppo debole per essere catturato con il campione attuale.

> <center> Limiti strutturali: campione di 82 giorni con sentiment ≠ 0 (sotto il minimo consigliato di 200–300); 18+ test multipli che aumentano il rischio di falsi positivi (con Bonferroni, soglia p < 0.003, solo il lead-lag a lag = −3 sarebbe significativo); nessun out-of-sample; frequenza irregolare del sentiment; singola fonte e singola metrica; GLD è un asset poco guidato dal sentiment retail.

> <center> Raccomandazioni: raccogliere almeno 2–3 anni di sentiment giornaliero continuo (500+ osservazioni); implementare validazione out-of-sample; aggiungere variabili di controllo (momentum, VIX); integrare più fonti (Twitter/X, Google Trends, news); testare su asset più reattivi al sentiment retail (BTC, meme stocks, ETF settoriali tech); applicare modelli non-lineari (Random Forest, Gradient Boosting).